## Import Polars

In [18]:
import polars as pl

## Read Data into DataFrame

In [19]:
COLUMN_NAMES = [
        "id", "price", "date", "postcode", "property_type",
        "old_new", "duration", "paon", "saon", "street",
        "locality", "town_city", "district", "county",
        "ppd_category", "record_type",
    ]

In [20]:
land_registry_data = pl.read_csv(
    source="../../data/land_registry_data/pp-*.csv",
    has_header=False,
    new_columns=COLUMN_NAMES,
    infer_schema=True,
    null_values=[""],
)

In [21]:
land_registry_data

id,price,date,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_type
str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""{3E0330F0-76C8-8D89-E050-A8C06…",103750,"""2016-09-21 00:00""","""CF62 6NT""","""F""","""N""","""L""","""COURTLANDS""","""FLAT 39""","""PARK ROAD""",null,"""BARRY""","""THE VALE OF GLAMORGAN""","""THE VALE OF GLAMORGAN""","""A""","""A"""
"""{3E0330F0-76C9-8D89-E050-A8C06…",165000,"""2016-09-16 00:00""","""NP10 9DN""","""S""","""N""","""F""","""10""",null,"""PARKLANDS CLOSE""","""ROGERSTONE""","""NEWPORT""","""NEWPORT""","""NEWPORT""","""A""","""A"""
"""{3E0330F0-76CA-8D89-E050-A8C06…",90000,"""2016-09-02 00:00""","""SA11 3SD""","""S""","""N""","""F""","""10""",null,"""BEACONS VIEW""",null,"""NEATH""","""NEATH PORT TALBOT""","""NEATH PORT TALBOT""","""A""","""A"""
"""{3E0330F0-76CB-8D89-E050-A8C06…",105000,"""2016-08-22 00:00""","""CF63 2QN""","""T""","""N""","""F""","""17""",null,"""NORDALE RISE""",null,"""BARRY""","""THE VALE OF GLAMORGAN""","""THE VALE OF GLAMORGAN""","""A""","""A"""
"""{3E0330F0-76CC-8D89-E050-A8C06…",67500,"""2016-08-30 00:00""","""NP18 1HA""","""S""","""N""","""F""","""11""",null,"""LAMB LANE""","""PONTHIR""","""NEWPORT""","""TORFAEN""","""TORFAEN""","""A""","""A"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""{36A61A95-56B0-DEF2-E063-4704A…",177500,"""2025-04-30 00:00""","""NR1 3SB""","""F""","""N""","""L""","""128""",null,"""BRAZEN GATE""",null,"""NORWICH""","""NORWICH""","""NORFOLK""","""B""","""A"""
"""{36A61A95-56B2-DEF2-E063-4704A…",113501,"""2025-05-02 00:00""","""PE30 5HJ""","""T""","""N""","""F""","""24""",null,"""SOUTH EVERARD STREET""",null,"""KING'S LYNN""","""KING'S LYNN AND WEST NORFOLK""","""NORFOLK""","""B""","""A"""
"""{36A61A95-56B5-DEF2-E063-4704A…",1100000,"""2025-04-24 00:00""","""NR16 1QH""","""O""","""N""","""F""","""HAY BARN HOUSE""",null,"""CARGATE COMMON""","""TIBENHAM""","""NORWICH""","""SOUTH NORFOLK""","""NORFOLK""","""B""","""A"""


In [22]:
land_registry_data = (
    land_registry_data
    .with_columns(
        pl.col("date").str.to_date(format="%Y-%m-%d %H:%M"),
    )
)

In [23]:
land_registry_data["property_type"].value_counts()

property_type,count
str,u32
"""T""",2696606
"""O""",582196
"""D""",2303875
"""S""",2625078
"""F""",1794725


In [24]:
land_registry_data = land_registry_data.with_columns(
    pl.col("property_type").replace({
        "D": "Detached",
        "S": "Semi-Detached",
        "T": "Terraced",
        "F": "Flat",
        "O": "Other",
    }).alias("property_type")
)

In [25]:
land_registry_data["property_type"].value_counts()

property_type,count
str,u32
"""Semi-Detached""",2625078
"""Terraced""",2696606
"""Other""",582196
"""Flat""",1794725
"""Detached""",2303875


In [26]:
land_registry_data = land_registry_data.filter(pl.col("property_type") != "Other")

In [27]:
land_registry_data = (
    land_registry_data
    .with_columns(
        pl.col("date").dt.year().alias("year"),
    )
)

In [28]:
land_registry_data.select(["id", "date", "year", "property_type", "price"])

id,date,year,property_type,price
str,date,i32,str,i64
"""{3E0330F0-76C8-8D89-E050-A8C06…",2016-09-21,2016,"""Flat""",103750
"""{3E0330F0-76C9-8D89-E050-A8C06…",2016-09-16,2016,"""Semi-Detached""",165000
"""{3E0330F0-76CA-8D89-E050-A8C06…",2016-09-02,2016,"""Semi-Detached""",90000
"""{3E0330F0-76CB-8D89-E050-A8C06…",2016-08-22,2016,"""Terraced""",105000
"""{3E0330F0-76CC-8D89-E050-A8C06…",2016-08-30,2016,"""Semi-Detached""",67500
…,…,…,…,…
"""{36A61A95-56AD-DEF2-E063-4704A…",2025-05-22,2025,"""Semi-Detached""",267500
"""{36A61A95-56AE-DEF2-E063-4704A…",2025-04-29,2025,"""Semi-Detached""",230000
"""{36A61A95-56B0-DEF2-E063-4704A…",2025-04-30,2025,"""Flat""",177500


In [29]:
annual_price_by_property_type = (
    land_registry_data
    .group_by(["year", "property_type"])
    .agg(
        pl.col("price").median().alias("median_price")
    )
    .sort(["year", "property_type"])
)

In [30]:
annual_price_by_property_type

year,property_type,median_price
i32,str,f64
2016,"""Detached""",306000.0
2016,"""Flat""",200000.0
2016,"""Semi-Detached""",187500.0
2016,"""Terraced""",169000.0
2017,"""Detached""",322500.0
…,…,…
2024,"""Terraced""",220000.0
2025,"""Detached""",416100.5
2025,"""Flat""",230000.0


## Visualise the results

In [31]:
import plotly.express as px

In [48]:
px.line(
    annual_price_by_property_type,
    x="year",
    y="median_price",
    color="property_type",
    markers=True,
    title="Median Price by Year and Property Type",
    width=800,
    height=600,
)

In [34]:
annual_price_by_property_type.write_delta("../../data/price_paid_insights/annual_price_by_property_type", mode="overwrite")

In [46]:
(
    pl.read_csv(
    source="../../data/land_registry_data/pp-*.csv",
    has_header=False,
    new_columns=COLUMN_NAMES,
    infer_schema=True,
    null_values=[""])
    .with_columns(
        pl.col("date").str.to_date(format="%Y-%m-%d %H:%M"),
    )
    .with_columns(
        pl.col("property_type").replace({
            "D": "Detached",
            "S": "Semi-Detached",
            "T": "Terraced",
            "F": "Flat",
            "O": "Other",
        }).alias("property_type")
    )
    .filter(pl.col("property_type") != "Other")
    .with_columns(
        pl.col("date").dt.year().alias("year")
    )
    .group_by(["year", "property_type"])
    .agg(
        pl.col("price").median().alias("median_price")
    )
    .sort(["year", "property_type"])
)

year,property_type,median_price
i32,str,f64
2016,"""Detached""",306000.0
2016,"""Flat""",200000.0
2016,"""Semi-Detached""",187500.0
2016,"""Terraced""",169000.0
2017,"""Detached""",322500.0
…,…,…
2024,"""Terraced""",220000.0
2025,"""Detached""",416100.5
2025,"""Flat""",230000.0


In [39]:
lazy_frame = (
    pl.scan_csv(
    source="../../data/land_registry_data/pp-*.csv",
    has_header=False,
    new_columns=COLUMN_NAMES,
    infer_schema=True,
    null_values=[""])
    .with_columns(
        pl.col("date").str.to_date(format="%Y-%m-%d %H:%M"),
    )
    .with_columns(
        pl.col("property_type").replace({
            "D": "Detached",
            "S": "Semi-Detached",
            "T": "Terraced",
            "F": "Flat",
            "O": "Other",
        }).alias("property_type")
    )
    .filter(pl.col("property_type") != "Other")
    .with_columns(
        pl.col("date").dt.year().alias("year")
    )
    .group_by(["year", "property_type"])
    .agg(
        pl.col("price").median().alias("median_price")
    )
    .sort(["year", "property_type"])
)

In [45]:
print(lazy_frame.explain(optimized=True))

SORT BY [col("year"), col("property_type")]
  AGGREGATE[maintain_order: false]
    [col("price").median().alias("median_price")] BY [col("year"), col("property_type")]
    FROM
     WITH_COLUMNS:
     [col("date").dt.year().alias("year")] 
      FILTER [(col("property_type")) != ("Other")]
      FROM
         WITH_COLUMNS:
         [col("date").str.strptime(["raise"]), col("property_type").replace([["D", "S", … "O"], ["Detached", "Semi-Detached", … "Other"]])] 
          Csv SCAN [../../data/land_registry_data/pp-2016.csv, ... 9 other sources]
          PROJECT 3/16 COLUMNS
          ESTIMATED ROWS: 1066856


In [47]:
lazy_frame.collect()

year,property_type,median_price
i32,str,f64
2016,"""Detached""",306000.0
2016,"""Flat""",200000.0
2016,"""Semi-Detached""",187500.0
2016,"""Terraced""",169000.0
2017,"""Detached""",322500.0
…,…,…
2024,"""Terraced""",220000.0
2025,"""Detached""",416100.5
2025,"""Flat""",230000.0
